# Abstract

- **Goal**: Develop a compact convolutional neural network (CNN) classifier for the MNIST dataset to achieve an accuracy score exceeding 90%.

- **Dataset**: [MNIST in CSV (Kaggle)](https://www.kaggle.com/datasets/oddrationale/mnist-in-csv)

- **Project Details**:  
  A custom CNN architecture is implemented, featuring two convolutional layers with ReLU activations, followed by max-pooling and a fully connected layer for digit classification. The model is trained using the Adam optimizer and cross-entropy loss on the MNIST dataset, split into training, validation, and test sets. An interactive form is included to enable users to select an image from the validation set and view the model's classification prediction.

- **Best Result**: The model achieved an accuracy of 0.984 on the test set, surpassing the target of 90% accuracy.

- **Sections**:  
  - [Imports](#Imports)  
  - [Model Parameters](#Model-Parameters)  
  - [Utils](#Utils)  
  - [Dataset](#Dataset)  
  - [Modeling](#Modeling)  
  - [Prediction](#Prediction)  
    - [Load Model](#Load-Model)  
    - [Setup Form](#Setup-Form)  
    - [Prediction Form](#Prediction-Form)

# Download & Install Dependencies

In [ ]:
# Install modules (if needed)
!pip install torch torchvision matplotlib pillow numpy tqdm ipywidgets ipython pandas

In [29]:
# Dataset Downloading
!mkdir data
!curl -L -o ./data/dataset.zip https://www.kaggle.com/api/v1/datasets/download/oddrationale/mnist-in-csv
!unzip -qq ./data/dataset.zip -d ./data
!rm ./data/dataset.zip
!rm ./data/mnist_test.csv
!mv ./data/mnist_train.csv ./data/mnist.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 15.2M  100 15.2M    0     0  6949k      0  0:00:02  0:00:02 --:--:-- 8683k


In [43]:
# Model Downloading
!mkdir models
from huggingface_hub import snapshot_download

PROJECT_NAME = "ImageClassificationMNIST"
MODEL_FOLDER = "torchconv"
repo_id = f"jsonmen/{PROJECT_NAME}"

snapshot_download(
    repo_id=repo_id,
    local_dir="./models",
    allow_patterns=[f"{MODEL_FOLDER}/*"],
    token=False  # No token needed for public repos
)
print(f"Downloaded models folder from {repo_id} to ./models")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

torchconv_model.pt:   0%|          | 0.00/105k [00:00<?, ?B/s]

Downloaded models folder from jsonmen/ImageClassificationMNIST to ./models


# Imports

In [2]:
import os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from torch import nn, Tensor, TensorType
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from torchvision.transforms import v2
import io

# Model Parameters

In [3]:
EPOCH = 3
BATCH_SIZE = 64
LR = 1e-3
INPUT_CHANNELS = 1
HIDDEN_CHANNELS = 16
NUM_CLASSES = 10
HIDDEN_DIM = 25
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Utils

In [4]:
def accuracy(y_pred, y_true):
    """
    Calculate the accuracy of predictions.

    Args:
    y_pred (torch.Tensor): Predicted labels (logits or probabilities).
    y_true (torch.Tensor): Ground truth labels (class indices or one-hot encoded).

    Returns:
    float: Accuracy of predictions.
    """
    # If y_true is one-hot encoded, convert it to class indices
    if y_true.dim() > 1 and y_true.size(1) > 1:
        y_true = torch.argmax(y_true, dim=1)

    # If y_pred is batched (2D tensor), get the predicted class by taking the argmax
    if y_pred.dim() > 1:
        y_pred = torch.argmax(y_pred, dim=1)

    # Calculate the number of correct predictions
    correct = (y_pred == y_true).float().sum()

    # Calculate the accuracy
    acc = correct / y_true.shape[0]

    return acc

In [5]:
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    """
    Train model for one epoch.
    """
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    progress_bar = tqdm(dataloader, desc="Train", leave=False)

    for inputs, targets in progress_bar:
        inputs = inputs.to(device)
        targets = targets.to(device).float()
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        acc = accuracy(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_acc += acc.item() * inputs.size(0)

        avg_loss = running_loss / ((progress_bar.n + 1) * inputs.size(0))
        avg_acc = running_acc / ((progress_bar.n + 1) * inputs.size(0))
        progress_bar.set_postfix(loss=avg_loss, acc=avg_acc)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc


def evaluate(model, dataloader, loss_fn, device):
    """
    Evaluate model.
    """
    model.eval()
    running_loss = 0.0
    running_acc = 0.0

    progress_bar = tqdm(dataloader, desc="Val", leave=False)

    with torch.no_grad():
        for inputs, targets in progress_bar:
            inputs = inputs.to(device)
            targets = targets.to(device).float()

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            acc = accuracy(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            running_acc += acc.item() * inputs.size(0)

            avg_loss = running_loss / ((progress_bar.n + 1) * inputs.size(0))
            avg_acc = running_acc / ((progress_bar.n + 1) * inputs.size(0))
            progress_bar.set_postfix(loss=avg_loss, acc=avg_acc)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc


def train_loop(model, train_loader, val_loader, optimizer, loss_fn, device, epochs):
    """
    Run full training loop.
    """
    # Example usage:
    # train_loop(
    #     model,
    #     train_loader,
    #     val_loader,
    #     optimizer,
    #     loss_fn,
    #     DEVICE,
    #     EPOCH
    # )
    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, loss_fn, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, loss_fn, device
        )

        print(
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

# Dataset

In [33]:
class MNISTConvDataset(Dataset):
    def __init__(self, dataset_path: str,  transform=None, num_classes: int = 10):
        self.dataset = pd.read_csv(dataset_path)
        self.num_classes = num_classes
    
    def __len__(self):
        return len(self.dataset)

    def onehot(self, idx):
        out = torch.zeros((self.num_classes, ))
        out[idx] = 1
        return out
    
    def __getitem__(self, idx: int):
        label = self.onehot(self.dataset.iloc[idx]["label"])
        image = torch.from_numpy(self.dataset.iloc[idx][self.dataset.columns[1:]].to_numpy().reshape(1, 28, 28))/255
        return image, label

# Modeling

In [34]:
class MNISTConvClassifier(nn.Module):
    def __init__(self, input_channels: int, hidden_channels: int, num_classes: int = 10, hidden_dim: int = 25):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(input_channels, hidden_channels, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2), stride=2),
            nn.Flatten(),
            nn.Linear(2304, num_classes),
        )
    def forward(self, x):
        return self.model(x)

In [35]:
dataset =  MNISTConvDataset("./data/mnist.csv")
train_dataset, val_dataset, test_dataset = random_split(dataset, [0.8, 0.1, 0.1])
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, drop_last=True, num_workers=2, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, drop_last=True, num_workers=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, drop_last=True, num_workers=2, shuffle=True)

In [36]:
model = MNISTConvClassifier(INPUT_CHANNELS, HIDDEN_CHANNELS, NUM_CLASSES, HIDDEN_DIM)
model = model.to(DEVICE)

In [37]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

In [38]:
train_loop(model, train_dataloader, val_dataloader, optimizer, loss_fn, DEVICE, EPOCH)


Epoch 1/3


Train Loss: 0.2567 | Train Acc: 0.9247 | Val Loss: 0.0938 | Val Acc: 0.9665

Epoch 2/3


Train Loss: 0.0729 | Train Acc: 0.9782 | Val Loss: 0.0725 | Val Acc: 0.9742

Epoch 3/3


Train Loss: 0.0518 | Train Acc: 0.9846 | Val Loss: 0.0577 | Val Acc: 0.9758


In [40]:
model.eval()

with torch.no_grad():
    test_loss = []
    test_acc = []
    for test_X, test_y in test_loader:
        test_X, test_y = test_X.to(DEVICE), test_y.to(DEVICE)
        test_pred = model(test_X)
        test_loss.append(loss_fn(test_pred, test_y))
        test_acc.append(accuracy(test_pred, test_y))

    print(f"Model Loss: {torch.mean(Tensor(test_loss)):.3f} Model Accuracy: {torch.mean(Tensor(test_acc)):.3f}")

Model Loss: 0.057 Model Accuracy: 0.982


In [41]:
torch.save(model.state_dict(), "./models/torchconv/torchconv_model.pt")

# Prediction

## Load Model

In [44]:
trained_model = MNISTConvClassifier(INPUT_CHANNELS, HIDDEN_CHANNELS, NUM_CLASSES, HIDDEN_DIM)
trained_model.load_state_dict(torch.load("./models/torchconv/torchconv_model.pt", map_location=torch.device('cpu')))
trained_model.cpu()
trained_model.eval()

MNISTConvClassifier(
  (model): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Flatten(start_dim=1, end_dim=-1)
    (6): Linear(in_features=2304, out_features=10, bias=True)
  )
)

## Setup Form

In [45]:
t1_output = widgets.Output()

image_id = widgets.BoundedIntText(
    value=7,
    min=0,
    max=len(val_dataset),
    step=1,
    description='Image id from dataset:',
    disabled=False,
    style={'description_width': 'initial'}
)
def t1_func(b):
    image_id_i = image_id.value
    with t1_output:
        clear_output()
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(val_dataset[image_id_i][0].permute(1, 2, 0).numpy())
        plt.title("Selected image")
        plt.xticks([])
        plt.yticks([])
        plt.subplot(1, 2, 2)
        plt.axis('off')
        prediction = trained_model(val_dataset[image_id_i][0].unsqueeze(0))
        info = f"""Number on image this is {torch.argmax(prediction).item()}"""

        plt.text(0, 1, info, fontsize=12, va='top')
        plt.tight_layout()
        plt.show()
        

t1_button = widgets.Button(description="Classify")
t1_button.on_click(t1_func)
form = widgets.VBox([image_id, t1_button, t1_output])

## Prediction Form

In [46]:
display(form)